# Sixty-beam OMEGA power deposition

This notebook runs the full spherical OMEGA example using the LILAC density and flow fits from Follett *et al.*, *Physics of Plasmas* **29**, 113902 (2022). It traces all 60 beams, constructs both caustic-resolved tetrahedral sheets, and conservatively deposits inverse-bremsstrahlung heating onto the hydro grid.

The default $12\times12$ rays per beam and 12 samples per sheet took about five minutes on the development CPU. JAX calls are explicitly synchronized below so the printed timings include execution rather than only asynchronous dispatch.

In [ ]:
from __future__ import annotations

import time
from dataclasses import replace
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm, Normalize
from matplotlib.patches import Circle

from pyGATH.fields import deposit_tetrahedral_power, tetrahedralise_sheet_fields
from pyGATH.grid import Geometry, convert_positions
from pyGATH.io import load_simulation_config
from pyGATH.plotting import plot_tetrahedral_mesh
from pyGATH.raytracing import RAY_STATE_LAYOUT, critical_density

plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})

## Resolution and plotting controls

Set `DEPOSITION_SCALE` to `"linear"` for shared linear colour limits. Increase `TETRAHEDRON_STRIDE` to thin the rendered edges without changing the calculation.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = (
    PROJECT_ROOT / "configs" / "example_configs" / "omega_60_beam_deposition.toml"
)
RAY_GRID = 12
SAMPLES_PER_SHEET = 12
TETRAHEDRON_STRIDE = 1
SLICE_RESOLUTION = 512
DEPOSITION_SCALE = "log"  # Change to "linear" if desired.
LOG_FLOOR_FRACTION = 1.0e-6
BEAM_INDEX = 0

if DEPOSITION_SCALE not in {"log", "linear"}:
    raise ValueError("DEPOSITION_SCALE must be 'log' or 'linear'")

## Timed simulation

The helper blocks on a representative JAX result for every accelerated stage. Tetrahedralisation includes its current CPU BVH construction.

In [ ]:
timings = {}


def timed_call(label, function, synchronize=None):
    started = time.perf_counter()
    value = function()
    if synchronize is not None:
        jax.block_until_ready(synchronize(value))
    elapsed = time.perf_counter() - started
    timings[label] = elapsed
    print(f"{label:24s}: {elapsed:9.3f} s")
    return value

In [ ]:
simulation = timed_call(
    "Load configuration", lambda: load_simulation_config(CONFIG_PATH)
)
simulation = replace(
    simulation,
    beams=replace(
        simulation.beams,
        nrays_axis1=RAY_GRID,
        nrays_axis2=RAY_GRID,
    ),
    raytracing=replace(simulation.raytracing, nsamples_per_sheet=SAMPLES_PER_SHEET),
)
grid = timed_call(
    "Build hydro grid", simulation.build_grid, lambda value: value.hydro.ne
)
beams = timed_call("Load 60 beams", simulation.load_beams, lambda value: value.origin)
initial_rays = timed_call(
    "Initialize ray tubes",
    lambda: simulation.initialize_rays(grid, beams),
    lambda value: value.state,
)

print(f"\nJAX device: {jax.devices()[0]}")
print(f"Hydro cells: {grid.ncells}")
print(f"Primary ray tubes: {beams.nbeams * RAY_GRID**2:,}")
print(f"Incident power: {float(initial_rays.total_incident_power) / 1e12:.6f} TW")

In [ ]:
trace = timed_call(
    "Trace and form sheets",
    lambda: simulation.trace_rays(initial_rays, grid),
    lambda value: value.sheet_fields,
)
print(f"All primary rays exited: {bool(trace.terminated)}")
print(f"Ray tubes with caustics: {int(np.asarray(trace.has_caustic).sum()):,}")

In [ ]:
tetrahedral_field = timed_call(
    "Tetrahedralise sheets",
    lambda: tetrahedralise_sheet_fields(
        trace.sheet_fields, fields="inverse_brems_deposition"
    ),
    lambda value: value.mesh.valid,
)
valid_by_sheet = np.asarray(tetrahedral_field.mesh.valid).sum(axis=(0, 2))
total_by_sheet = tetrahedral_field.mesh.nbeams * tetrahedral_field.mesh.ntetrahedra
for sheet_index, valid_count in enumerate(valid_by_sheet):
    print(
        f"Sheet {sheet_index + 1}: {int(valid_count):,} / "
        f"{total_by_sheet:,} valid tetrahedra"
    )

In [ ]:
deposition_options = simulation.extra_sections.get("deposition", {})
deposition = timed_call(
    "Deposit onto hydro grid",
    lambda: deposit_tetrahedral_power(
        tetrahedral_field,
        grid,
        max_subdivision_levels=int(deposition_options.get("max_subdivision_levels", 1)),
        relative_tolerance=float(deposition_options.get("relative_tolerance", 1.0e-2)),
        tetrahedron_batch_size=int(
            deposition_options.get("tetrahedron_batch_size", 2048)
        ),
    ),
    lambda value: value.power_density,
)

In [ ]:
incident_power = float(initial_rays.total_incident_power)
source_power = float(deposition.source_power)
deposited_power = float(deposition.deposited_power)
outside_power = float(deposition.outside_power)

print("Power balance")
print(f"  Incident:             {incident_power / 1e12:12.6f} TW")
print(f"  Reconstructed source: {source_power / 1e12:12.6f} TW")
print(f"  Deposited in grid:     {deposited_power / 1e12:12.6f} TW")
print(f"  Outside grid:          {outside_power / 1e12:12.6f} TW")
print(f"  Deposited / incident:  {deposited_power / incident_power:12.6%}")
print(f"  Scatter residual:      {float(deposition.conservation_error):12.6e} W")

## Timings

In [ ]:
figure, axis = plt.subplots(figsize=(9, 4))
labels = list(timings)
seconds = np.asarray([timings[label] for label in labels])
axis.barh(labels, seconds, color="tab:blue")
axis.set_xlabel("Wall time [s]")
axis.set_title(f"OMEGA {RAY_GRID}x{RAY_GRID} rays, {SAMPLES_PER_SHEET} samples/sheet")
axis.invert_yaxis()
for index, value in enumerate(seconds):
    axis.text(value, index, f" {value:.2f} s", va="center")
figure.tight_layout()

## Beam 1 tetrahedral sheets

The detected caustic vertices are overlaid in red. Rendering can be thinned using `TETRAHEDRON_STRIDE` without altering the field or deposition calculation.

In [ ]:
figure = plt.figure(figsize=(16, 7))
caustic_mask = np.asarray(trace.has_caustic[BEAM_INDEX])
caustic_positions = np.asarray(
    trace.sheet_fields[BEAM_INDEX, 0, ..., -1, RAY_STATE_LAYOUT.position]
)[caustic_mask]

for sheet_index in range(2):
    axis = figure.add_subplot(1, 2, sheet_index + 1, projection="3d")
    plot_tetrahedral_mesh(
        tetrahedral_field,
        beam_index=BEAM_INDEX,
        sheet_index=sheet_index,
        tetrahedron_stride=TETRAHEDRON_STRIDE,
        ax=axis,
    )
    if caustic_positions.size:
        axis.scatter(
            *caustic_positions.T,
            color="red",
            marker="x",
            s=20,
            linewidth=0.8,
            label="Caustic vertices",
        )
        axis.legend(loc="upper right")
    axis.view_init(elev=22, azim=-55)
    axis.set_title(f"Beam {beams.names[BEAM_INDEX]}, sheet {sheet_index + 1}")
figure.suptitle("Tetrahedral beam volumes")
figure.tight_layout()

## Cartesian power-deposition slices

The conservative result is cell centred on a spherical mesh. For visualization only, each Cartesian pixel below takes the value of its containing spherical hydro cell; no smoothing is applied.

In [ ]:
def cartesian_cell_slice(grid, cell_values, projection, resolution):
    component_pairs = {"xy": (0, 1), "yz": (1, 2), "xz": (0, 2)}
    first_component, second_component = component_pairs[projection]
    limit = float(grid.xb[-1])
    edges = np.linspace(-limit, limit, resolution + 1)
    centres = 0.5 * (edges[:-1] + edges[1:])
    first, second = np.meshgrid(centres, centres, indexing="xy")
    cartesian = np.zeros((*first.shape, 3), dtype=np.float64)
    cartesian[..., first_component] = first
    cartesian[..., second_component] = second
    native = np.asarray(
        convert_positions(jnp.asarray(cartesian), Geometry.CARTESIAN, grid.geom)
    )

    boundaries = tuple(np.asarray(axis) for axis in (grid.xb, grid.yb, grid.zb))
    indices = [
        np.searchsorted(axis, native[..., component], side="right") - 1
        for component, axis in enumerate(boundaries)
    ]
    inside = np.ones(first.shape, dtype=bool)
    for component, axis in enumerate(boundaries):
        inside &= native[..., component] >= axis[0]
        inside &= native[..., component] <= axis[-1]
        indices[component] = np.clip(indices[component], 0, axis.size - 2)

    sampled = np.full(first.shape, np.nan, dtype=np.float64)
    sampled[inside] = np.asarray(cell_values)[
        indices[0][inside], indices[1][inside], indices[2][inside]
    ]
    return edges, sampled

In [ ]:
projections = ("xy", "yz", "xz")
slice_data = {
    projection: cartesian_cell_slice(
        grid, deposition.power_density, projection, SLICE_RESOLUTION
    )
    for projection in projections
}
positive_values = np.concatenate(
    [values[np.isfinite(values) & (values > 0.0)] for _, values in slice_data.values()]
)
maximum = float(positive_values.max())
if DEPOSITION_SCALE == "log":
    minimum = max(float(positive_values.min()), maximum * LOG_FLOOR_FRACTION)
    colour_norm = LogNorm(vmin=minimum, vmax=maximum)
else:
    colour_norm = Normalize(vmin=0.0, vmax=maximum)

ncritical = float(critical_density(beams.omega[0]))
radial_density_ratio = np.asarray(grid.hydro.ne[:, 0, 0]) / ncritical
critical_radius = np.exp(
    np.interp(
        0.0,
        np.log(radial_density_ratio[::-1]),
        np.log(np.asarray(grid.xb)[::-1]),
    )
)

figure, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)
last_mesh = None
for axis, projection in zip(axes, projections, strict=True):
    edges, values = slice_data[projection]
    plotted_values = np.ma.masked_invalid(values)
    if DEPOSITION_SCALE == "log":
        plotted_values = np.ma.masked_less_equal(plotted_values, 0.0)
    edges_um = edges * 1.0e6
    last_mesh = axis.pcolormesh(
        edges_um,
        edges_um,
        plotted_values,
        shading="flat",
        norm=colour_norm,
        cmap="inferno",
    )
    axis.add_patch(
        Circle(
            (0.0, 0.0),
            critical_radius * 1.0e6,
            fill=False,
            color="cyan",
            linestyle="--",
            linewidth=1.2,
            label="critical surface",
        )
    )
    axis.set_aspect("equal")
    axis.set_xlabel(f"{projection[0]} [um]")
    axis.set_ylabel(f"{projection[1]} [um]")
    axis.set_title(f"{projection[0]}-{projection[1]} plane")
    axis.legend(loc="upper right")
colourbar = figure.colorbar(last_mesh, ax=axes, shrink=0.92)
colourbar.set_label(r"Power deposition $Q_{IB}$ [W m$^{-3}$]")
figure.suptitle(f"OMEGA inverse-bremsstrahlung deposition ({DEPOSITION_SCALE} scale)")

## Radially integrated deposition

In [ ]:
radial_power = np.asarray(deposition.cell_power).sum(axis=(1, 2))
radial_centres = np.asarray(grid.xc)
radial_widths = np.diff(np.asarray(grid.xb))
cumulative = np.cumsum(radial_power) / radial_power.sum()

figure, power_axis = plt.subplots(figsize=(9, 4.5))
power_axis.bar(
    radial_centres * 1.0e6,
    radial_power / 1.0e12,
    width=radial_widths * 1.0e6,
    color="tab:red",
    alpha=0.75,
    label="Power per shell",
)
power_axis.axvline(
    critical_radius * 1.0e6, color="black", linestyle="--", label="Critical surface"
)
power_axis.set_xlabel("Radius [um]")
power_axis.set_ylabel("Deposited power per shell [TW]")
power_axis.legend(loc="upper left")

cumulative_axis = power_axis.twinx()
cumulative_axis.plot(
    radial_centres * 1.0e6, cumulative, color="tab:blue", linewidth=2.0
)
cumulative_axis.set_ylabel("Cumulative deposited-power fraction", color="tab:blue")
cumulative_axis.set_ylim(0.0, 1.02)
power_axis.set_title("Radial inverse-bremsstrahlung deposition")
figure.tight_layout()